# 第 4 天：保持神经网络简洁（MDL）

Hinton 与 van Camp（1993）：《通过最小化权重描述长度保持神经网络简洁》（Keeping Neural Networks Simple by Minimizing the Description Length of the Weights）。

论文联系了**高斯权重分布**、描述长度最小化与变分贝叶斯推断。在相应假设下，它们得到相同的优化目标。原仓库据此说明含噪权重可促使网络寻找更平坦、稳健的极小值；平坦性与泛化之间并非无条件等价。

## 本 Notebook 的内容

1. **带缺口的正弦波**：在缺少一段区域的数据上训练。
2. **训练贝叶斯网络**：损失由误差与 KL 散度（复杂度）组成。
3. **不确定性可视化**：多次前向传播，观察模型在哪些区域较确定或较不确定。
4. **权重分布**：查看哪些权重需要较低 sigma，哪些可能成为剪枝候选；不能仅凭 sigma 大小断言可无损剪枝。

## 核心思想

普通权重是点估计（w = 5.123），MDL 权重是分布（w ~ N(5.1, 0.2)）。如果网络在权重带噪时仍能正常工作，就不必保留那么高的精度；降低精度可以减少描述开销。

> 原论文 PDF 本次访问返回 HTTP 403，具体实验和部分历史归属待核验。本 Notebook 是教学实现，不是原论文实验的直接复现。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# 将当前目录加入模块搜索路径
sys.path.insert(0, str(Path.cwd()))

from implementation import MDLNetwork
from visualization import (plot_loss_dynamics, plot_uncertainty_envelope,
                           plot_weight_distributions, plot_snr_analysis,
                           analyze_compression_stats)

print("所有模块导入成功。")

## 1. 数据：带缺口的正弦波

为了理解贝叶斯权重的作用，需要能体现“自信”预测与合理不确定性之间差异的数据集。

生成正弦波后，**删除中间部分**（|x| < 1）。普通神经网络可能在缺口处仍给出确定预测；贝叶斯网络预期在缺口处表现出更高不确定性，因为未见过该区域的数据。这是待观察的实验现象，不是任意贝叶斯近似都能保证的性质。


In [ ]:
# 生成带缺口的含噪正弦波
def generate_gappy_data(n=100):
    # 左侧（-3 至 -1）
    X1 = np.random.uniform(-3, -1, n//2)
    # 右侧（1 至 3）
    X2 = np.random.uniform(1, 3, n//2)
    # 合并
    X = np.concatenate([X1, X2])
    # 添加噪声
    y = np.sin(X) + np.random.normal(0, 0.1, n)
    return X.reshape(-1, 1), y.reshape(-1, 1)

X_train, y_train = generate_gappy_data(100)

# 可视化范围（包含缺口）
X_test = np.linspace(-4, 4, 200).reshape(-1, 1)

plt.figure(figsize=(10, 5))
plt.scatter(X_train, y_train, c='red', label='训练数据')
plt.axvspan(-1, 1, color='gray', alpha=0.2, label='缺口（无数据）')
plt.plot(X_test, np.sin(X_test), 'k--', alpha=0.5, label='真实正弦波')
plt.title("挑战：预测缺口中的函数值")
plt.legend()
plt.show()

## 2. 训练贝叶斯网络

MDL 网络有**两个**损失项，对应论文核心目标的教学形式：

1. **误差代价（NLL）**：预测对数据的拟合程度。
2. **复杂度代价（KL）**：描述权重的信息代价；代码使用自然对数，单位为奈特（nat），转换为比特需除以 ln 2。

总损失 = 误差 + beta * KL

`kl_weight`（beta）控制权衡。过小可能过拟合、不确定性不足；过大可能欠拟合、忽略数据。合适取值旨在反映数据缺失处的不确定性，仍需验证校准。

> **原有实现不一致：** 下一单元显示的 KL 除以样本数，但传给 `update_weights` 的 KL 权重没有除以样本数，因此日志目标与参数更新的 KL 缩放不一致。本次保留算法，仅记录问题。


In [ ]:
# 初始化网络
net = MDLNetwork(input_size=1, hidden_size=20, output_size=1)

# 超参数
epochs = 2000
lr = 0.01
kl_weight = 0.1  # “简洁性压力”

# 存储绘图数据
history = {'total': [], 'nll': [], 'kl': []}

print("训练贝叶斯网络……")
print("=" * 50)

for epoch in range(epochs):
    # 1. 前向传播（随机采样权重）
    # 每次调用时，网络权重都会略有不同。
    preds = net.forward(X_train)
    
    # 2. 数据损失（用 MSE 代替负对数似然的比例形式）
    nll = np.mean((preds - y_train)**2)
    d_nll = 2 * (preds - y_train) / len(X_train)
    
    # 3. 复杂度损失（KL 散度）
    kl = net.total_kl() / len(X_train)
    
    # 4. 总损失
    loss = nll + kl_weight * kl
    
    # 存储历史记录
    history['total'].append(loss)
    history['nll'].append(nll)
    history['kl'].append(kl)
    
    # 5. 反向传播
    net.backward(d_nll)
    
    # 6. 更新权重
    net.update_weights(lr, kl_weight)
    
    if epoch % 200 == 0:
        print(f"轮次 {epoch:4d} | 总损失：{loss:.4f} | 误差：{nll:.4f} | 复杂度：{kl:.4f}")

print("\n训练完成。")

## 3. 损失动态：复杂度与误差

观察训练中两个损失项如何相互作用。
**误差（NLL）** 随拟合数据而迅速下降；**复杂度（KL）** 可能在权重偏离先验时先上升，随后在网络找到紧凑解时趋于稳定。


In [ ]:
plot_loss_dynamics(history)

## 4. 不确定性包络

这是贝叶斯神经网络的典型可视化。

在测试数据上运行网络 **100 次**。权重由分布表示，因此每次前向传播采样不同权重，预测结果略有差异。

* **存在训练数据的区域：** 预期预测较集中，方差较低。
* **缺口区域：** 预期预测较分散，方差较高，以反映缺少数据。

这种分散程度来自高斯权重分布，表示模型的认知不确定性（epistemic uncertainty）。实际表现依赖近似分布与训练，不保证自动校准良好。


In [ ]:
plot_uncertainty_envelope(net, X_train, y_train, X_test, n_samples=100)

## 5. 权重分布

论文标题讨论“最小化权重描述长度”，下面观察它在实现中的表现。

每个权重具有学习得到的 sigma（标准差）：
* **较小 sigma：** 网络需要更精确的权重值，通常需要更多精度描述开销。
* **较大 sigma：** 权重允许更大波动，精度要求较低。

相对于均值而言 sigma 较大（信噪比较低）的权重可作为剪枝候选。

> **原文技术限制：** 原仓库将大 sigma 或低信噪比等同于“几乎零比特”或“已经压缩掉”，这种等同不成立。实际 KL 同时取决于均值、标准差和先验，过大的 sigma 也会增加 KL；这里没有实际执行编码。


In [ ]:
plot_weight_distributions(net, 'layer1')
plot_snr_analysis(net)

## 6. 压缩统计

用信噪比（Signal-to-Noise Ratio，SNR = |mu|/sigma）进行启发式分析。
原仓库将低于阈值（例如 0.5）的权重视为可置零的候选；是否不损害模型，必须用剪枝后的实际误差验证，不能只凭 SNR 保证。这里的压缩数值是估计，不是实际编码文件大小。


In [ ]:
analyze_compression_stats(net, threshold_snr=0.5)

## 7. 关键收获

**1. 通过压缩促进泛化（论文核心论证）：**
KL 惩罚限制描述权重所需的信息。无助于解释数据的权重分布被拉回先验；原仓库将其直观类比为剪枝。MDL 寻求拟合数据与模型描述复杂度的平衡，而不是保证任何情况下都得到最简单或最能泛化的网络。

**2. 表达不确定性（该建模形式的用途）：**
普通网络可能在缺口处给出确定但错误的曲线。MDL 网络通过权重分布产生不同预测，以表达“见过数据”与“只能猜测”的差异。是否真的在缺口处增大不确定性，仍需检查结果。

**3. 训练噪声具有建模作用：**
训练中从分布采样权重，与比特回收论证所考虑的随机权重相联系。原仓库引用 Hinton 与 van Camp（1993）第 2 节说明噪声允许降低编码代价；具体章节尚待原文核验。重参数化采样本身并不构成一个实际的比特回收编码器。

---

**下一篇：** [第 5 天：最小描述长度原理](../05_MDL_Principle/README.md)。原文误写为指针网络，现按仓库实际目录修正。
